# Demo 2 — Tono puro e suono complesso

Al primo avvio usare **Run → Run All Cells** (▶▶): la preparazione carica `audiolab`. Poi modificare la cella della demo e rieseguirla con **Shift + Invio**. Ogni riquadro è indipendente.

Le funzioni `al.…` sono quelle della libreria del corso. La prima esecuzione richiede il download del runtime e delle dipendenze.

In [ ]:
# Preparazione automatica: la stessa libreria audiolab dei laboratori.
import sys
from pathlib import Path
if sys.platform == "emscripten":
    import micropip
    await micropip.install(["numpy", "scipy", "soundfile", "matplotlib", "plotly==6.3.0", "nbformat>=5.10"])
    from pyodide.http import pyfetch
    from js import location
    import io, zipfile
    ROOT = Path("/home/pyodide/infomm-3485c94980f2f6df")
    if not (ROOT / "lab-2026/audiolab/__init__.py").exists():
        prefisso = "/infomm-audio-AI" if location.hostname.endswith("github.io") else ""
        url = str(location.origin) + prefisso + "/_jupyterlite/files/audiolab-course.zip?v=3485c94980f2f6df"
        risposta = await pyfetch(url)
        if not risposta.ok:
            raise RuntimeError("Impossibile caricare audiolab: " + str(risposta.status))
        with zipfile.ZipFile(io.BytesIO(await risposta.bytes())) as archivio:
            archivio.extractall(ROOT)
else:
    ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
                 if (p / "lab-2026/audiolab").is_dir()), None)
    if ROOT is None:
        raise FileNotFoundError("Avviare Jupyter dalla cartella InfoMM-local del corso.")
sys.path.insert(0, str(ROOT / "lab-2026"))
import audiolab as al
import numpy as np
import matplotlib.pyplot as plt
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots
from scipy import signal
from IPython.display import Audio, Markdown, display
import time
pio.renderers.default = "plotly_mimetype"
AUDIO = al.AUDIO_DIR
DEMO_ASSETS = ROOT / "lab-2026/interattivo/assets"
SR = FS = 22050
N_FFT, HOP = 2048, 512
BLU, ARANCIO, VERDE = "#1766A5", "#D86822", "#277668"
print("audiolab", al.__version__, "— ambiente pronto")


In [ ]:
f0 = 220.0
t = np.arange(2 * SR) / SR
puro = np.sin(2 * np.pi * f0 * t)
complesso = sum(np.sin(2 * np.pi * k * f0 * t) / k for k in range(1, 7))
puro *= 0.15 / np.sqrt(np.mean(puro ** 2))
complesso *= 0.15 / np.sqrt(np.mean(complesso ** 2))
fig = make_subplots(rows=3, cols=2, subplot_titles=("tono puro", "sei armoniche",
                                          "spettro lineare", "spettro lineare",
                                          "livello [dB]", "livello [dB]"))
for j, (x, titolo) in enumerate([(puro, "tono puro"), (complesso, "sei armoniche")]):
    fig.add_trace(go.Scatter(x=t[:441] * 1000, y=x[:441], mode="lines",
                             line=dict(width=1, color="#1769C2"), showlegend=False), row=1, col=j + 1)
    f, amp = al.amplitude_spectrum(x[SR // 2:SR // 2 + SR], SR, window="boxcar")
    selezione = (f <= 1500) & (amp > 1e-8)
    fig.add_trace(go.Bar(x=f[selezione], y=amp[selezione], marker_color="#1769C2",
                         showlegend=False), row=2, col=j + 1)
    fig.add_trace(go.Scatter(x=f[selezione], y=20 * np.log10(np.maximum(amp[selezione], 1e-8)),
                             mode="lines", line=dict(width=1, color="#B8860B"),
                             showlegend=False), row=3, col=j + 1)
fig.update_xaxes(range=[0, 1500], row=2, col=1); fig.update_xaxes(range=[0, 1500], row=2, col=2)
fig.update_xaxes(range=[0, 1500], row=3, col=1); fig.update_xaxes(range=[0, 1500], row=3, col=2)
fig.update_yaxes(range=[-0.4, 0.4], row=1, col=1)
fig.update_layout(height=760, title="Tono puro e suono complesso: forma d'onda, spettro lineare e in dB")
fig.show()
